In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np

from src.data import DataLoader
from src.timescales import Timescales

In [ ]:
# Select the dataset to process.
# Available: 'HCP-YA', 'HCP-D', 'WAND', 'Penn_LEAD'
dataset = 'HCP-YA'

# Optional parcellation override. Leave as None to use the dataset-specific default
# from DATASET_PARAMS (e.g. 'Schaefer4007'). Set to e.g. 'Schaefer10007' to process
# at a different resolution. This determines the seg- token written into the scan
# registry, and therefore the parcellation the timescale outputs are saved/loaded at.
seg = None

# HCP-YA only: resample the 3T timeseries (rest and task) onto the 7T sampling grid
# (target TR pulled from the config, MRI7T = 1.0s) at load time. When True, every
# ACF / timescale / metric output, the timeseries dumps, and the scan registry are
# tagged '_resampled' so resampled and native outputs coexist. Ignored for non-HCP-YA.
resample_hcpya_3t = False

data_loader = DataLoader(dataset=dataset)

# Load fMRI data

In [ ]:
from src.config import get_dataset_params

# ── Load parameters for the selected dataset ─────────────────────────────────
# Edit DATASET_PARAMS in src/notebook.py to change processing parameters.
# Pass seg=seg to override the dataset's default parcellation (None keeps the default).
params = get_dataset_params(dataset, seg=seg)
detrend             = params['detrend']
nuisance_regression = params['nuisance_regression']
task_regression     = params['task_regression']

subject_ids = data_loader.load_subject_ids(params['subject_ids_file'])
# subject_ids = subject_ids[:10]  # limit for testing

print(f"Dataset:    {dataset}")
print(f"Modalities: {params['modalities']}")
print(f"Sessions: {params['sessions']}")
print(f"Seg: {params['seg']}")
print(f"Detrend: {detrend}")
print(f"Nuis. regression: {nuisance_regression}")
print(f"Task regression: {task_regression}")

print(f"Subjects:   {len(subject_ids)}")
subject_ids[:5]

In [ ]:
for modality in params['modalities']:
    data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=params['sessions'],
        tasks=params['tasks'][modality],
        space=params['space'],
        seg=params['seg'],
        modality=modality,
    )

print(f"Registered {len(data_loader.scan_registry)} scans across {len(params['modalities'])} modality(ies).")

In [ ]:
data_loader.validate_scan_registry(
    verbose=True,
    remove_incomplete_subjects=params['remove_incomplete_subjects'],
)

# Optionally resample HCP-YA 3T onto the 7T (1.0s) grid at load time. This rewrites the
# tr of every MRI3T registry scan to the target, so cache keys, the exported registry,
# and every output filename reflect the resampled grid. resample_tag feeds the
# '_resampled' token into the compute calls, timeseries dumps, and registry filename.
resample_active = bool(resample_hcpya_3t) and dataset == 'HCP-YA'
if resample_active:
    target_tr = data_loader.enable_hcpya_3t_resampling('MRI7T')
    print(f"Resampling HCP-YA 3T timeseries -> {target_tr}s at load time; outputs tagged '_resampled'.")
resample_tag = '_resampled' if resample_active else ''
# Prefix form, placed before the frequency-filter token so the stat name mirrors
# the processing order (resample -> filter): e.g. timescale_..._resampled_bp-0.01-0.08hz.
resample_prefix = 'resampled_' if resample_active else ''

In [ ]:
timeseries_data = data_loader.load_batch(
    detrend=detrend,
    nuisance_regression=nuisance_regression,
    frequency_filter=False,
    verbose=True,
)

# The XCP-D 4S456Parcels atlas returns the 400 Schaefer-400 (7Networks) cortical
# parcels followed by 56 subcortical/cerebellar parcels. Keep the cortical parcels
# only, so PNC matches the Schaefer-400 convention used by every other dataset and
# the S-A axis (verified: 4S456 columns 0:400 == schaefer400-7 order).
if params['seg'] == '4S456Parcels':
    timeseries_data = {
        scan_string: (ts[:400] if ts is not None else None)
        for scan_string, ts in timeseries_data.items()
    }

In [ ]:
# The registry filename encodes the dataset and, for HCP-YA, the active modality.
active_modality = params['modalities'][0] if len(params['modalities']) == 1 else 'multi'
modality_suffix = {
    'MRI3T': '',
    'MRI7T': '_7T',
    'MEG':   '_MEG',
    'multi': '_multi',
}.get(active_modality, f'_{active_modality}')

# Append the parcellation only when seg was explicitly overridden, so the default
# resolution keeps the original filename (e.g. scan_registry_HCP-D.csv).
seg_suffix = f'_{seg}' if seg is not None else ''

registry_filename = f'scan_registry_{dataset}{modality_suffix}{seg_suffix}{resample_tag}.csv'
data_loader.export_scan_registry(registry_filename)

scan_registry = data_loader.list_registered_scans()
scan_registry.head()

In [ ]:
subject_ids = scan_registry['sub'].unique()
n_subs = len(subject_ids)
print(n_subs)

In [ ]:
subject_ids

# Process fMRI timescales

In [ ]:
run_cell = True

In [ ]:
if run_cell:
    timescales_processor = Timescales(dataset=dataset, timeseries_data=timeseries_data, verbose=True)
    if task_regression:
        timescales_processor.regress_task_structure()

### Save out time series

for others 

In [ ]:
if run_cell:
    out_dir = os.path.join(data_loader.output_dir, 'data', 'timeseries', dataset)
    os.makedirs(out_dir, exist_ok=True)

    seg = params['seg']
    tasks = sorted(set(
        part[len('task-'):]
        for k in timeseries_data
        for part in k.split('_')
        if part.startswith('task-')
    ))

    for task in tasks:
        task_ts = {k: v for k, v in timeseries_data.items() if f'task-{task}_' in k}
        if not task_ts:
            continue
        out_path = os.path.join(out_dir, f'{dataset}_{seg}_task-{task}{resample_tag}_data-ts.npy')
        np.save(out_path, task_ts, allow_pickle=True)
        print(f"  {task}: {len(task_ts)} scans  →  {out_path}")

        if task_regression:
            task_resid = {k: v for k, v in timescales_processor.timeseries_residuals.items()
                          if f'task-{task}_' in k and v is not None}
            if task_resid:
                out_path = os.path.join(out_dir, f'{dataset}_{seg}_task-{task}{resample_tag}_data-tsresid.npy')
                np.save(out_path, task_resid, allow_pickle=True)
                print(f"  {task}: {len(task_resid)} residuals  →  {out_path}")

In [ ]:
if run_cell and task_regression:
    out_dir = os.path.join(data_loader.output_dir, 'data', 'task_contrasts', dataset)
    os.makedirs(out_dir, exist_ok=True)

    seg = params['seg']
    tasks = sorted(set(
        part[len('task-'):]
        for k in timescales_processor.task_contrasts
        for part in k.split('_')
        if part.startswith('task-')
    ))

    for task in tasks:
        task_data = {k: v for k, v in timescales_processor.task_contrasts.items()
                     if f'task-{task}_' in k and v is not None}
        if not task_data:
            continue
        out_path = os.path.join(out_dir, f'{dataset}_{seg}_task-{task}_data-task_contrasts.npy')
        np.save(out_path, task_data, allow_pickle=True)
        print(f"  {task}: {len(task_data)} scans  →  {out_path}")

### Timeseries metrics (not timescales)

In [ ]:
if run_cell:
    for method in ['tsnr', 'var']:
        timescales_processor.compute_timeseries_measures(method=method, resampled=resample_active, force_recompute=False, verbose=True)

        if task_regression:
            timescales_processor.compute_timeseries_measures(method=method, use_residuals=True, resampled=resample_active, force_recompute=False, verbose=True)

### Timescale measures

In [ ]:
timescale_measures = [
    'auc_zero_crossing',
    'lag_one',
    'auc_half_crossing',
]

In [ ]:
if run_cell:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, resampled=resample_active, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=False, suffix=(resample_tag.lstrip('_') or None), force_recompute=False, verbose=True)
        
    autocorr_dict_raw = timescales_processor.autocorr_dict.copy()
    timescales_dict_raw = timescales_processor.timescales_dict.copy()

In [ ]:
if run_cell and task_regression:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, resampled=resample_active, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix=('resampled_resids' if resample_active else 'resids'), force_recompute=False, verbose=True)
        
    autocorr_dict_resid = timescales_processor.autocorr_dict.copy()
    timescales_dict_resid = timescales_processor.timescales_dict.copy()

### Frequency specific timescales

In [ ]:
run_cell = True

In [ ]:
if run_cell and dataset == 'HCP-D':
    filter_configs = [
        {'lowpass': 0.08, 'highpass': 0.01},  # bandpass (standard)
        # {'lowpass': 0.15, 'highpass': 0.08},  # bandpass (higher freqs)
        # {'lowpass': None, 'highpass': 0.01},  # highpass (standard)
        # {'lowpass': 0.08, 'highpass': None},  # lowpass (standard)
    ]

    for cfg in filter_configs:
        print(cfg)
        lowpass  = cfg['lowpass']
        highpass = cfg['highpass']

        if highpass is not None and lowpass is not None:
            suffix = f'bp-{highpass}-{lowpass}hz'
        elif highpass is not None:
            suffix = f'hp-{highpass}hz'
        elif lowpass is not None:
            suffix = f'lp-{lowpass}hz'

        # Load with filtering applied before task regression
        timeseries_data_filt = data_loader.load_batch(
            detrend=detrend,
            nuisance_regression=nuisance_regression,
            frequency_filter=True,
            lowpass=lowpass,
            highpass=highpass,
            verbose=True,
        )
        
        # filter cortex
        if params['seg'] == '4S456Parcels':
            timeseries_data_filt = {
                scan_string: (ts[:400] if ts is not None else None)
                for scan_string, ts in timeseries_data_filt.items()
            }

        processor_filt = Timescales(dataset=dataset, timeseries_data=timeseries_data_filt, verbose=False)
        if task_regression:
            processor_filt.regress_task_structure()

        # lowpass/highpass passed for output naming only; filtering already applied at load time
        processor_filt.compute_autocorrelation_function(use_fft=True, use_residuals=False, resampled=resample_active, force_recompute=False,
                                                        lowpass=lowpass, highpass=highpass, verbose=False)
        for timescale_measure in timescale_measures:
            processor_filt.compute_timescale_measures(method=timescale_measure, use_residuals=False,
                                                      suffix=f'{resample_prefix}{suffix}', force_recompute=False, verbose=False)

        if task_regression:
            processor_filt.compute_autocorrelation_function(use_fft=True, use_residuals=True, resampled=resample_active, force_recompute=False,
                                                            lowpass=lowpass, highpass=highpass, verbose=False)
            for timescale_measure in timescale_measures:
                processor_filt.compute_timescale_measures(method=timescale_measure, use_residuals=True,
                                                          suffix=f'{resample_prefix}{suffix}_resids', force_recompute=False, verbose=False)